# Домашнє завдання до теми 4 - DML, DDL та складні SQL-вирази

## 1. Setup і PostgreSQL
Якщо `pgserver` не підтримується поточною версією Python у Colab, використайте runtime Python 3.12.

In [25]:
%pip install -q pgserver psycopg2-binary sqlalchemy pandas kagglehub

import sys
from pathlib import Path
import pandas as pd
import pgserver
import kagglehub
from sqlalchemy import create_engine, text
from IPython.display import display, Markdown

print('Python:', sys.version)
pg = pgserver.get_server('/tmp/hw4_pg', cleanup_mode='stop')
engine = create_engine(pg.get_uri(), future=True)
with engine.connect() as conn:
    print(conn.execute(text('SELECT version()')).scalar_one())

def execute_script(sql):
    raw = engine.raw_connection()
    try:
        with raw.cursor() as cur:
            cur.execute(sql)
        raw.commit()
    finally:
        raw.close()

def run_query(sql, params=None):
    return pd.read_sql(text(sql), engine, params=params or {})

def scalar(sql):
    with engine.connect() as conn:
        return conn.execute(text(sql)).scalar_one()

Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
PostgreSQL 16.2 on x86_64-pc-linux-gnu, compiled by gcc (GCC) 10.2.1 20210130 (Red Hat 10.2.1-11), 64-bit


## 2. Отримання Olist
Використовується публічний Kaggle dataset `olistbr/brazilian-ecommerce`. Потрібні CSV знаходяться рекурсивно за назвою.

In [26]:
dataset_dir = Path(kagglehub.dataset_download('olistbr/brazilian-ecommerce'))
print('Dataset directory:', dataset_dir)

files = {
    'olist_customers_raw': 'olist_customers_dataset.csv',
    'olist_orders_raw': 'olist_orders_dataset.csv',
    'olist_order_items_raw': 'olist_order_items_dataset.csv',
    'olist_products_raw': 'olist_products_dataset.csv',
    'olist_sellers_raw': 'olist_sellers_dataset.csv',
    'olist_order_reviews_raw': 'olist_order_reviews_dataset.csv',
}
resolved = {}
for table_name, file_name in files.items():
    matches = list(dataset_dir.rglob(file_name))
    if not matches:
        raise FileNotFoundError(file_name)
    resolved[table_name] = matches[0]
    print(table_name, '->', matches[0])

Using Colab cache for faster access to the 'brazilian-ecommerce' dataset.
Dataset directory: /kaggle/input/brazilian-ecommerce
olist_customers_raw -> /kaggle/input/brazilian-ecommerce/olist_customers_dataset.csv
olist_orders_raw -> /kaggle/input/brazilian-ecommerce/olist_orders_dataset.csv
olist_order_items_raw -> /kaggle/input/brazilian-ecommerce/olist_order_items_dataset.csv
olist_products_raw -> /kaggle/input/brazilian-ecommerce/olist_products_dataset.csv
olist_sellers_raw -> /kaggle/input/brazilian-ecommerce/olist_sellers_dataset.csv
olist_order_reviews_raw -> /kaggle/input/brazilian-ecommerce/olist_order_reviews_dataset.csv


## 3. Raw-рівень
Raw-таблиці відображають CSV і використовуються як проміжний шар.

In [27]:
raw_shapes=[]
for table_name, path in resolved.items():
    df=pd.read_csv(path)
    df.to_sql(table_name, engine, if_exists='replace', index=False, chunksize=10000)
    raw_shapes.append((table_name, len(df), len(df.columns)))
display(pd.DataFrame(raw_shapes, columns=['table_name','n_rows','n_columns']))

,table_name,n_rows,n_columns
0,olist_customers_raw,99441,5
1,olist_orders_raw,99441,8
2,olist_order_items_raw,112650,7
3,olist_products_raw,32951,9
4,olist_sellers_raw,3095,4
5,olist_order_reviews_raw,99224,7


## 4. Typed-схема Olist
`review_id` не використовується як безумовний primary key. Для reviews створюється технічний `review_row_id`.

In [28]:
typed_ddl = r"""
DROP TABLE IF EXISTS customer_segments CASCADE;
DROP TABLE IF EXISTS seller_score CASCADE;
DROP TABLE IF EXISTS seller_alerts CASCADE;
DROP TABLE IF EXISTS olist_customer_dim CASCADE;
DROP TABLE IF EXISTS olist_order_reviews CASCADE;
DROP TABLE IF EXISTS olist_order_items CASCADE;
DROP TABLE IF EXISTS olist_orders CASCADE;
DROP TABLE IF EXISTS olist_customers CASCADE;
DROP TABLE IF EXISTS olist_products CASCADE;
DROP TABLE IF EXISTS olist_sellers CASCADE;

CREATE TABLE olist_customers (
 customer_id TEXT PRIMARY KEY,
 customer_unique_id TEXT NOT NULL,
 customer_zip_code_prefix INTEGER,
 customer_city TEXT,
 customer_state CHAR(2) NOT NULL
);
CREATE INDEX idx_olist_customers_unique_id ON olist_customers(customer_unique_id);

CREATE TABLE olist_orders (
 order_id TEXT PRIMARY KEY,
 customer_id TEXT NOT NULL REFERENCES olist_customers(customer_id),
 order_status TEXT NOT NULL,
 order_purchase_timestamp TIMESTAMP NOT NULL,
 order_approved_at TIMESTAMP,
 order_delivered_carrier_date TIMESTAMP,
 order_delivered_customer_date TIMESTAMP,
 order_estimated_delivery_date TIMESTAMP,
 CHECK (order_status IN ('created','approved','invoiced','processing','shipped','delivered','unavailable','canceled'))
);

CREATE TABLE olist_products (
 product_id TEXT PRIMARY KEY,
 product_category_name TEXT,
 product_name_length INTEGER CHECK (product_name_length IS NULL OR product_name_length >= 0),
 product_description_length INTEGER CHECK (product_description_length IS NULL OR product_description_length >= 0),
 product_photos_qty INTEGER CHECK (product_photos_qty IS NULL OR product_photos_qty >= 0),
 product_weight_g INTEGER CHECK (product_weight_g IS NULL OR product_weight_g >= 0),
 product_length_cm INTEGER CHECK (product_length_cm IS NULL OR product_length_cm >= 0),
 product_height_cm INTEGER CHECK (product_height_cm IS NULL OR product_height_cm >= 0),
 product_width_cm INTEGER CHECK (product_width_cm IS NULL OR product_width_cm >= 0)
);

CREATE TABLE olist_sellers (
 seller_id TEXT PRIMARY KEY,
 seller_zip_code_prefix INTEGER,
 seller_city TEXT,
 seller_state CHAR(2) NOT NULL
);

CREATE TABLE olist_order_items (
 order_id TEXT NOT NULL REFERENCES olist_orders(order_id),
 order_item_id INTEGER NOT NULL,
 product_id TEXT REFERENCES olist_products(product_id),
 seller_id TEXT REFERENCES olist_sellers(seller_id),
 shipping_limit_date TIMESTAMP,
 price NUMERIC(12,2) CHECK (price IS NULL OR price >= 0),
 freight_value NUMERIC(12,2) CHECK (freight_value IS NULL OR freight_value >= 0),
 PRIMARY KEY(order_id, order_item_id)
);
CREATE INDEX idx_olist_order_items_seller_order ON olist_order_items(seller_id, order_id);

CREATE TABLE olist_order_reviews (
 review_row_id BIGINT GENERATED BY DEFAULT AS IDENTITY PRIMARY KEY,
 review_id TEXT,
 order_id TEXT NOT NULL REFERENCES olist_orders(order_id),
 review_score SMALLINT CHECK (review_score BETWEEN 1 AND 5),
 review_comment_title TEXT,
 review_comment_message TEXT,
 review_creation_date TIMESTAMP,
 review_answer_timestamp TIMESTAMP
);
CREATE INDEX idx_olist_order_reviews_order_id ON olist_order_reviews(order_id);
"""
execute_script(typed_ddl)
print('Typed-схему створено.')

Typed-схему створено.


## 5. Raw -> typed через INSERT ... SELECT

In [29]:
typed_load = r"""
INSERT INTO olist_customers
SELECT customer_id, customer_unique_id, customer_zip_code_prefix::INTEGER, customer_city, customer_state::CHAR(2)
FROM olist_customers_raw;

INSERT INTO olist_orders
SELECT order_id, customer_id, order_status, order_purchase_timestamp::TIMESTAMP,
       order_approved_at::TIMESTAMP, order_delivered_carrier_date::TIMESTAMP,
       order_delivered_customer_date::TIMESTAMP, order_estimated_delivery_date::TIMESTAMP
FROM olist_orders_raw;

INSERT INTO olist_products
SELECT product_id, product_category_name, product_name_lenght::INTEGER,
       product_description_lenght::INTEGER, product_photos_qty::INTEGER,
       product_weight_g::INTEGER, product_length_cm::INTEGER,
       product_height_cm::INTEGER, product_width_cm::INTEGER
FROM olist_products_raw;

INSERT INTO olist_sellers
SELECT seller_id, seller_zip_code_prefix::INTEGER, seller_city, seller_state::CHAR(2)
FROM olist_sellers_raw;

INSERT INTO olist_order_items
SELECT order_id, order_item_id::INTEGER, product_id, seller_id,
       shipping_limit_date::TIMESTAMP, price::NUMERIC(12,2), freight_value::NUMERIC(12,2)
FROM olist_order_items_raw;

INSERT INTO olist_order_reviews
(review_id, order_id, review_score, review_comment_title, review_comment_message, review_creation_date, review_answer_timestamp)
SELECT review_id, order_id, review_score::SMALLINT, review_comment_title, review_comment_message,
       review_creation_date::TIMESTAMP, review_answer_timestamp::TIMESTAMP
FROM olist_order_reviews_raw;
"""
execute_script(typed_load)
print('Typed-таблиці наповнено.')

Typed-таблиці наповнено.


In [30]:
smoke=run_query("""
SELECT 'customers' table_name,(SELECT COUNT(*) FROM olist_customers_raw) raw_rows,(SELECT COUNT(*) FROM olist_customers) typed_rows
UNION ALL SELECT 'orders',(SELECT COUNT(*) FROM olist_orders_raw),(SELECT COUNT(*) FROM olist_orders)
UNION ALL SELECT 'order_items',(SELECT COUNT(*) FROM olist_order_items_raw),(SELECT COUNT(*) FROM olist_order_items)
UNION ALL SELECT 'products',(SELECT COUNT(*) FROM olist_products_raw),(SELECT COUNT(*) FROM olist_products)
UNION ALL SELECT 'sellers',(SELECT COUNT(*) FROM olist_sellers_raw),(SELECT COUNT(*) FROM olist_sellers)
UNION ALL SELECT 'reviews',(SELECT COUNT(*) FROM olist_order_reviews_raw),(SELECT COUNT(*) FROM olist_order_reviews)
ORDER BY table_name
""")
display(smoke)
display(Markdown(f"**Інтерпретація:** однакова кількість raw і typed rows для всіх таблиць: **{bool((smoke.raw_rows==smoke.typed_rows).all())}**."))

,table_name,raw_rows,typed_rows
0,customers,99441,99441
1,order_items,112650,112650
2,orders,99441,99441
3,products,32951,32951
4,reviews,99224,99224
5,sellers,3095,3095


**Інтерпретація:** однакова кількість raw і typed rows для всіх таблиць: **True**.

# 6. DML-запити PostgreSQL

## 6.1 INSERT ... RETURNING
Створюємо alert для seller з найбільшою кількістю запізнілих доставок. `RETURNING` одразу показує створений рядок і може бути audit-output.

In [31]:
execute_script(r"""
DROP TABLE IF EXISTS seller_alerts CASCADE;
CREATE TABLE seller_alerts (
 alert_id BIGINT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
 seller_id TEXT NOT NULL REFERENCES olist_sellers(seller_id),
 alert_type TEXT NOT NULL CHECK (alert_type IN ('late_delivery','low_rating','data_quality')),
 severity SMALLINT NOT NULL CHECK (severity BETWEEN 1 AND 5),
 details TEXT,
 created_at TIMESTAMPTZ NOT NULL DEFAULT NOW()
);
""")
q=r"""
WITH seller_orders AS (
 SELECT DISTINCT seller_id, order_id FROM olist_order_items
), late_sellers AS (
 SELECT so.seller_id, COUNT(DISTINCT o.order_id) late_orders
 FROM seller_orders so JOIN olist_orders o ON o.order_id=so.order_id
 WHERE o.order_delivered_customer_date IS NOT NULL
   AND o.order_estimated_delivery_date IS NOT NULL
   AND o.order_delivered_customer_date::DATE > o.order_estimated_delivery_date::DATE
 GROUP BY so.seller_id
 ORDER BY late_orders DESC, so.seller_id
 LIMIT 1
)
INSERT INTO seller_alerts(seller_id,alert_type,severity,details)
SELECT seller_id,'late_delivery',4,'Seller has '||late_orders||' late delivered orders in the loaded dataset'
FROM late_sellers
RETURNING alert_id,seller_id,alert_type,severity,created_at;
"""
with engine.begin() as conn:
    r=conn.execute(text(q)); rows=r.fetchall(); cols=r.keys()
display(pd.DataFrame(rows,columns=cols))

,alert_id,seller_id,alert_type,severity,created_at
0,1,4a3ca9315b744ce9f8e9374361493884,late_delivery,4,2026-10-18 09:50:22.147943+00:00


Цей окремий INSERT не є idempotent при багаторазовому ручному запуску клітинки. Відтворюваність повного notebook забезпечує повторне створення `seller_alerts` перед INSERT. `RETURNING` дає audit-result без окремого SELECT.

## 6.2 ON CONFLICT DO UPDATE - seller_score
Перед агрегацією review дані приводяться до грануляності `seller_id x order_id`, щоб не множити review score через item-level join.

In [32]:
execute_script(r"""
DROP TABLE IF EXISTS seller_score CASCADE;
CREATE TABLE seller_score (
 seller_id TEXT PRIMARY KEY REFERENCES olist_sellers(seller_id),
 avg_rating NUMERIC(3,2),
 n_reviews INTEGER NOT NULL DEFAULT 0 CHECK(n_reviews>=0),
 tier TEXT CHECK(tier IN ('gold','silver','bronze')),
 updated_at TIMESTAMPTZ NOT NULL DEFAULT NOW()
);
""")
q=r"""
WITH seller_orders AS (
 SELECT DISTINCT seller_id,order_id FROM olist_order_items
), review_per_order AS (
 SELECT order_id,AVG(review_score)::NUMERIC(3,2) order_review_score
 FROM olist_order_reviews WHERE review_score IS NOT NULL GROUP BY order_id
), seller_rating AS (
 SELECT so.seller_id,AVG(rpo.order_review_score)::NUMERIC(3,2) avg_rating,
        COUNT(rpo.order_id)::INTEGER n_reviews
 FROM seller_orders so LEFT JOIN review_per_order rpo ON rpo.order_id=so.order_id
 GROUP BY so.seller_id
)
INSERT INTO seller_score(seller_id,avg_rating,n_reviews,tier)
SELECT seller_id,avg_rating,n_reviews,
 CASE WHEN avg_rating IS NULL THEN NULL WHEN avg_rating>=4.5 THEN 'gold' WHEN avg_rating>=3.5 THEN 'silver' ELSE 'bronze' END
FROM seller_rating
ON CONFLICT(seller_id) DO UPDATE
SET avg_rating=EXCLUDED.avg_rating,n_reviews=EXCLUDED.n_reviews,tier=EXCLUDED.tier,updated_at=NOW()
RETURNING seller_id,avg_rating,n_reviews,tier,updated_at;
"""
with engine.begin() as conn:
    r=conn.execute(text(q)); sample=r.fetchmany(20); cols=r.keys(); r.close()
display(pd.DataFrame(sample,columns=cols))
display(run_query("SELECT COUNT(*) n_sellers,COUNT(*) FILTER(WHERE avg_rating IS NULL) sellers_without_rating,COUNT(*) FILTER(WHERE tier='gold') gold,COUNT(*) FILTER(WHERE tier='silver') silver,COUNT(*) FILTER(WHERE tier='bronze') bronze FROM seller_score"))

,seller_id,avg_rating,n_reviews,tier,updated_at
0,a61cc04793308395a840807104365121,5.00,1,gold,2026-10-18 09:50:22.306201+00:00
1,6d2f2e3b539480db1e0842b3a4e32e6e,3.67,3,silver,2026-10-18 09:50:22.306201+00:00
2,c53bcd3be457a342a97e39e5a9f0be22,3.20,5,bronze,2026-10-18 09:50:22.306201+00:00
3,9591fc341b1bfb7ef561e2968ec6e011,1.00,1,bronze,2026-10-18 09:50:22.306201+00:00
4,f9ec7093df3a7b346b7bcf7864069ca3,4.78,9,gold,2026-10-18 09:50:22.306201+00:00
5,fc38b5dceee1a730fad8853453437fbd,4.60,5,gold,2026-10-18 09:50:22.306201+00:00
6,5def4c3732941a971cba8fdee992ede1,4.60,5,gold,2026-10-18 09:50:22.306201+00:00
7,b39d7fe263ef469605dbb32608aee0af,3.88,43,silver,2026-10-18 09:50:22.306201+00:00
8,ea65d8b58316a6f2362f2a9e4b3e86ad,4.00,9,silver,2026-10-18 09:50:22.306201+00:00
9,a45765f8afb1e594b22bf9e974b46765,4.00,1,silver,2026-10-18 09:50:22.306201+00:00


,n_sellers,sellers_without_rating,gold,silver,bronze
0,3095,5,983,1568,539


`SELECT DISTINCT` у `seller_orders` прибирає повтори одного seller-order через кілька order items. `review_per_order` агрегує reviews до `order_id`. `ON CONFLICT` робить upsert повторюваним без дублювання seller rows. Seller без review може мати `avg_rating=NULL`, `n_reviews=0`, `tier=NULL`. Review Olist належить order, а не окремому seller, тому multi-seller order є навчальним спрощенням.

## 6.3 UPDATE ... FROM - is_late

In [33]:
execute_script("ALTER TABLE olist_orders ADD COLUMN IF NOT EXISTS is_late BOOLEAN NOT NULL DEFAULT FALSE;")
q=r"""
WITH delivery_flags AS (
 SELECT order_id,(order_delivered_customer_date IS NOT NULL AND order_estimated_delivery_date IS NOT NULL
 AND order_delivered_customer_date::DATE > order_estimated_delivery_date::DATE) is_late_calc
 FROM olist_orders
)
UPDATE olist_orders o SET is_late=f.is_late_calc
FROM delivery_flags f WHERE f.order_id=o.order_id
RETURNING o.order_id,o.customer_id,o.is_late;
"""
with engine.begin() as conn:
    r=conn.execute(text(q)); sample=r.fetchmany(20); cols=r.keys(); r.close()
display(pd.DataFrame(sample,columns=cols))
display(run_query("SELECT COUNT(*) n_orders,COUNT(*) FILTER(WHERE is_late) n_late,ROUND(COUNT(*) FILTER(WHERE is_late)*100.0/NULLIF(COUNT(*),0),2) pct_late FROM olist_orders"))

,order_id,customer_id,is_late
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,False
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,False
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,False
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,False
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,False
5,a4591c265e18cb1dcee52889e2d8acc3,503740e9ca751ccdda7ba28e9ab8f608,False
6,136cce7faa42fdb2cefd53fdc79a6098,ed0271e0b7da060a393796590e7b737a,False
7,6514b8ad8028c9f2cc2374ded245783f,9bdf08b4b3b52b5526ff42d37d47f222,False
8,76c6e866289321a7c93b82b54852dc33,f54a9f0e6b351c431402b8461ea51999,False
9,e69bfb5eb88e0ed6a785585b27e16dbf,31ad1d1b63eb9962463f764d4e6e0c9d,False


,n_orders,n_late,pct_late
0,99441,6535,6.57


Операція idempotent: повторний запуск знову обчислює `is_late` з поточних дат доставки.

## 6.4 DELETE ... RETURNING - старі alert-и

In [34]:
q="DELETE FROM seller_alerts WHERE created_at < NOW()-INTERVAL '30 days' RETURNING alert_id,seller_id,alert_type,created_at"
with engine.begin() as conn:
    r=conn.execute(text(q)); rows=r.fetchall(); cols=r.keys()
deleted=pd.DataFrame(rows,columns=cols); display(deleted)
display(Markdown(f"**Інтерпретація:** видалено **{len(deleted)}** старих alert-ів. Нульовий результат допустимий, бо alert створено щойно."))

,alert_id,seller_id,alert_type,created_at


**Інтерпретація:** видалено **0** старих alert-ів. Нульовий результат допустимий, бо alert створено щойно.

# 7. olist_customer_dim і customer_segments
Сегментація виконується на рівні `customer_unique_id`, який краще відповідає реальному покупцю.

In [35]:
execute_script(r"""
DROP TABLE IF EXISTS customer_segments CASCADE;
DROP TABLE IF EXISTS olist_customer_dim CASCADE;
CREATE TABLE olist_customer_dim (
 customer_unique_id TEXT PRIMARY KEY,
 latest_customer_id TEXT REFERENCES olist_customers(customer_id),
 customer_state CHAR(2),
 customer_city TEXT
);
INSERT INTO olist_customer_dim(customer_unique_id,latest_customer_id,customer_state,customer_city)
SELECT DISTINCT ON(c.customer_unique_id) c.customer_unique_id,c.customer_id,c.customer_state,c.customer_city
FROM olist_customers c LEFT JOIN olist_orders o ON o.customer_id=c.customer_id
ORDER BY c.customer_unique_id,o.order_purchase_timestamp DESC NULLS LAST,c.customer_id;

CREATE TABLE customer_segments (
 segment_id BIGINT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
 customer_unique_id TEXT NOT NULL,
 segment_name TEXT NOT NULL CHECK(segment_name IN ('VIP','regular','new','inactive')),
 rfm_score SMALLINT NOT NULL CHECK(rfm_score BETWEEN 1 AND 5),
 recency_days INTEGER CHECK(recency_days IS NULL OR recency_days>=0),
 monetary_value NUMERIC(12,2) NOT NULL DEFAULT 0 CHECK(monetary_value>=0),
 n_orders INTEGER NOT NULL DEFAULT 0 CHECK(n_orders>=0),
 assigned_at TIMESTAMPTZ NOT NULL DEFAULT NOW(),
 UNIQUE(customer_unique_id)
);
ALTER TABLE customer_segments ADD CONSTRAINT fk_customer_segments_customer_unique
FOREIGN KEY(customer_unique_id) REFERENCES olist_customer_dim(customer_unique_id) ON DELETE CASCADE;
""")

In [36]:
q=r"""
WITH reference_date AS (
 SELECT MAX(order_purchase_timestamp)::DATE+1 as_of_date FROM olist_orders
), order_totals AS (
 SELECT o.order_id,o.customer_id,o.order_purchase_timestamp::DATE order_date,
 COALESCE(SUM(COALESCE(oi.price,0)+COALESCE(oi.freight_value,0)),0)::NUMERIC(12,2) order_total
 FROM olist_orders o LEFT JOIN olist_order_items oi ON oi.order_id=o.order_id
 GROUP BY o.order_id,o.customer_id,o.order_purchase_timestamp
), customer_stats AS (
 SELECT c.customer_unique_id,COUNT(DISTINCT ot.order_id)::INTEGER n_orders,
 COALESCE(SUM(ot.order_total),0)::NUMERIC(12,2) monetary_value,MAX(ot.order_date) last_order_date
 FROM olist_customer_dim cd JOIN olist_customers c ON c.customer_unique_id=cd.customer_unique_id
 LEFT JOIN order_totals ot ON ot.customer_id=c.customer_id
 GROUP BY c.customer_unique_id
), segmented AS (
 SELECT cs.customer_unique_id,cs.n_orders,cs.monetary_value,
 CASE WHEN cs.last_order_date IS NULL THEN NULL ELSE (rd.as_of_date-cs.last_order_date)::INTEGER END recency_days,
 CASE WHEN cs.monetary_value>=1000 THEN 'VIP'
      WHEN cs.last_order_date IS NOT NULL AND (rd.as_of_date-cs.last_order_date)::INTEGER<=90 AND cs.n_orders=1 THEN 'new'
      WHEN cs.n_orders>=2 OR cs.monetary_value>=100 THEN 'regular' ELSE 'inactive' END segment_name,
 CASE WHEN cs.monetary_value>=1000 THEN 5 WHEN cs.monetary_value>=500 THEN 4 WHEN cs.n_orders>=2 THEN 3 WHEN cs.n_orders=1 THEN 2 ELSE 1 END::SMALLINT rfm_score
 FROM customer_stats cs CROSS JOIN reference_date rd
)
INSERT INTO customer_segments(customer_unique_id,segment_name,rfm_score,recency_days,monetary_value,n_orders)
SELECT customer_unique_id,segment_name,rfm_score,recency_days,monetary_value,n_orders FROM segmented
ON CONFLICT(customer_unique_id) DO UPDATE
SET segment_name=EXCLUDED.segment_name,rfm_score=EXCLUDED.rfm_score,recency_days=EXCLUDED.recency_days,
    monetary_value=EXCLUDED.monetary_value,n_orders=EXCLUDED.n_orders,assigned_at=NOW()
RETURNING customer_unique_id,segment_name,rfm_score,recency_days,monetary_value,n_orders;
"""
with engine.begin() as conn:
    r=conn.execute(text(q)); sample=r.fetchmany(20); cols=r.keys(); r.close()
display(pd.DataFrame(sample,columns=cols))
display(run_query("SELECT segment_name,COUNT(*) n_customers,AVG(monetary_value)::NUMERIC(12,2) avg_monetary,AVG(n_orders)::NUMERIC(8,2) avg_orders FROM customer_segments GROUP BY segment_name ORDER BY n_customers DESC"))

,customer_unique_id,segment_name,rfm_score,recency_days,monetary_value,n_orders
0,0000366f3b9a7992bf8c76cfdf3221e2,regular,2,161,141.90,1
1,0000b849f77a49e4a4ce2b2a4ca5be3f,inactive,2,164,27.19,1
2,0000f46a3911fa3c0805444483337064,inactive,2,587,86.22,1
3,0000f6ccb0745a6a4b88665a16c9f078,inactive,2,371,43.62,1
4,0004aac84e0df4da2b147fca70cf8255,regular,2,338,196.89,1
5,0004bd2a26a76fe21f786e4fbd80607f,regular,2,196,166.98,1
6,00050ab1314c0e55a6ca13cf7181fecf,inactive,2,181,35.38,1
7,00053a61a98854899e70ed204dd4bafe,regular,2,232,419.18,1
8,0005e1862207bf6ccc02e4228effd9a0,regular,2,593,150.12,1
9,0005ef4cd20d2893f0d9fbd94d3c0d97,regular,2,220,129.76,1


,segment_name,n_customers,avg_monetary,avg_orders
0,regular,45982,224.57,1.07
1,inactive,40032,59.28,1.00
2,new,8891,140.39,1.00
3,VIP,1191,1591.81,1.11


In [37]:
display(run_query("SELECT conname,contype,pg_get_constraintdef(oid) definition FROM pg_constraint WHERE conrelid='customer_segments'::regclass ORDER BY conname"))

,conname,contype,definition
0,customer_segments_customer_unique_id_key,u,UNIQUE (customer_unique_id)
1,customer_segments_monetary_value_check,c,CHECK ((monetary_value >= (0)::numeric))
2,customer_segments_n_orders_check,c,CHECK ((n_orders >= 0))
3,customer_segments_pkey,p,PRIMARY KEY (segment_id)
4,customer_segments_recency_days_check,c,CHECK (((recency_days IS NULL) OR (recency_day...
5,customer_segments_rfm_score_check,c,CHECK (((rfm_score >= 1) AND (rfm_score <= 5)))
6,customer_segments_segment_name_check,c,"CHECK ((segment_name = ANY (ARRAY['VIP'::text,..."
7,fk_customer_segments_customer_unique,f,FOREIGN KEY (customer_unique_id) REFERENCES ol...


`olist_customer_dim` дає стабільний FK на `customer_unique_id`. `reference_date` береться з самого історичного датасету, тому recency залишається відтворюваною. `ON CONFLICT` дозволяє безпечно перераховувати сегментацію.

# 8. JOIN-запити

## JOIN 1 - INNER JOIN
**Business-question:** які category + seller state дають найбільший revenue у delivered orders?

In [38]:
q1=run_query("""
SELECT p.product_category_name,s.seller_state,COUNT(DISTINCT oi.order_id) n_orders,
SUM(oi.price+oi.freight_value)::NUMERIC(12,2) revenue
FROM olist_order_items oi JOIN olist_orders o ON o.order_id=oi.order_id
JOIN olist_products p ON p.product_id=oi.product_id JOIN olist_sellers s ON s.seller_id=oi.seller_id
WHERE o.order_status='delivered'
GROUP BY p.product_category_name,s.seller_state ORDER BY revenue DESC NULLS LAST LIMIT 10
"""); display(q1)
if len(q1):
 r=q1.iloc[0]; display(Markdown(f"**Інтерпретація:** лідер Top-10 - **{r.product_category_name} / {r.seller_state}**, revenue **{r.revenue}**, orders **{r.n_orders}**."))

,product_category_name,seller_state,n_orders,revenue
0,cama_mesa_banho,SP,8214,1070048.14
1,relogios_presentes,SP,4494,1016168.80
2,beleza_saude,SP,5684,790967.66
3,esporte_lazer,SP,4772,680165.54
4,moveis_decoracao,SP,4597,605650.63
5,cool_stuff,SP,2621,484196.53
6,utilidades_domesticas,SP,4021,471810.66
7,automotivo,SP,3018,438833.22
8,informatica_acessorios,SP,2783,388506.42
9,bebes,SP,1981,339129.74


**Інтерпретація:** лідер Top-10 - **cama_mesa_banho / SP**, revenue **1070048.14**, orders **8214**.

## JOIN 2 - LEFT JOIN + COALESCE
**Business-question:** які customer-level features можна побудувати на рівні `customer_unique_id` без множення review score?

In [39]:
q2=run_query("""
WITH order_totals AS (
 SELECT o.order_id,c.customer_unique_id,COALESCE(SUM(COALESCE(oi.price,0)+COALESCE(oi.freight_value,0)),0)::NUMERIC(12,2) order_total
 FROM olist_orders o JOIN olist_customers c ON c.customer_id=o.customer_id
 LEFT JOIN olist_order_items oi ON oi.order_id=o.order_id GROUP BY o.order_id,c.customer_unique_id
), review_per_order AS (
 SELECT order_id,AVG(review_score)::NUMERIC(3,2) order_review_score FROM olist_order_reviews
 WHERE review_score IS NOT NULL GROUP BY order_id
), customer_stats AS (
 SELECT ot.customer_unique_id,COUNT(DISTINCT ot.order_id) n_orders,COALESCE(SUM(ot.order_total),0)::NUMERIC(12,2) total_spend,
 AVG(rpo.order_review_score)::NUMERIC(3,2) avg_review_score
 FROM order_totals ot LEFT JOIN review_per_order rpo ON rpo.order_id=ot.order_id GROUP BY ot.customer_unique_id
)
SELECT cd.customer_unique_id,cd.customer_state,COALESCE(cs.n_orders,0) n_orders,
COALESCE(cs.total_spend,0)::NUMERIC(12,2) total_spend,COALESCE(cs.avg_review_score,0)::NUMERIC(3,2) avg_review_score
FROM olist_customer_dim cd LEFT JOIN customer_stats cs ON cs.customer_unique_id=cd.customer_unique_id
ORDER BY total_spend DESC NULLS LAST LIMIT 20
"""); display(q2)
if len(q2):
 r=q2.iloc[0]; display(Markdown(f"**Інтерпретація:** найбільший total spend у sample має `{r.customer_unique_id}`: **{r.total_spend}**, orders **{r.n_orders}**. `review_per_order` захищає rating від множення через items."))

,customer_unique_id,customer_state,n_orders,total_spend,avg_review_score
0,0a0a92112bd4c708ca5fde585afaa872,RJ,1,13664.08,1.0
1,da122df9eeddfedc1dc1f5349a1a690c,RJ,2,7571.63,5.0
2,763c8b1c9c68a0229c42c9fc6f662b93,ES,1,7274.88,1.0
3,dc4802a71eae9be1dd28f5d788ceb526,MS,1,6929.31,5.0
4,459bef486812aa25204be022145caa62,ES,1,6922.21,0.0
5,ff4159b92c40ebe40454e3e6a7c35ed6,SP,1,6726.66,5.0
6,4007669dec559734d6f53e029e360987,MG,1,6081.54,1.0
7,5d0a2980b292d049061542014e8960bf,GO,1,4809.44,1.0
8,eebb5dda148d3893cdaf5b5ca3040ccb,SP,1,4764.34,4.0
9,48e1ac109decbb87765a3eade6854098,PB,1,4681.78,5.0


**Інтерпретація:** найбільший total spend у sample має `0a0a92112bd4c708ca5fde585afaa872`: **13664.08**, orders **1**. `review_per_order` захищає rating від множення через items.

## JOIN 3 - FULL OUTER JOIN
**Business-question:** які states є тільки серед customers, тільки серед sellers або в обох наборах?

In [40]:
q3=run_query("""
WITH c AS (SELECT customer_state state,COUNT(*) n_customers FROM olist_customers GROUP BY customer_state),
s AS (SELECT seller_state state,COUNT(*) n_sellers FROM olist_sellers GROUP BY seller_state)
SELECT COALESCE(c.state,s.state) state,c.n_customers,s.n_sellers,
CASE WHEN c.state IS NOT NULL AND s.state IS NOT NULL THEN 'both' WHEN c.state IS NOT NULL THEN 'customers_only' ELSE 'sellers_only' END state_presence
FROM c FULL OUTER JOIN s ON s.state=c.state ORDER BY state
"""); display(q3)
display(Markdown(f"**Інтерпретація:** distribution state_presence = **{q3.state_presence.value_counts().to_dict()}**. FULL OUTER JOIN не втрачає states, які є лише з одного боку."))

,state,n_customers,n_sellers,state_presence
0,AC,81,1.0,both
1,AL,413,NaN,customers_only
2,AM,148,1.0,both
3,AP,68,NaN,customers_only
4,BA,3380,19.0,both
5,CE,1336,13.0,both
6,DF,2140,30.0,both
7,ES,2033,23.0,both
8,GO,2020,40.0,both
9,MA,747,1.0,both


**Інтерпретація:** distribution state_presence = **{'both': 23, 'customers_only': 4}**. FULL OUTER JOIN не втрачає states, які є лише з одного боку.

## JOIN 4 - SELF JOIN
**Business-question:** які seller-пари з одного штату, але різних міст можна порівняти регіонально?

In [41]:
q4=run_query("""
SELECT s1.seller_id seller_a,s2.seller_id seller_b,s1.seller_state,s1.seller_city city_a,s2.seller_city city_b
FROM olist_sellers s1 JOIN olist_sellers s2 ON s1.seller_state=s2.seller_state
AND s1.seller_id<s2.seller_id AND s1.seller_city IS DISTINCT FROM s2.seller_city
ORDER BY s1.seller_state,seller_a,seller_b LIMIT 20
"""); display(q4)
display(Markdown(f"**Інтерпретація:** показано **{len(q4)}** пар. Умова `s1.seller_id < s2.seller_id` прибирає self-пари та дзеркальні дублікати."))

,seller_a,seller_b,seller_state,city_a,city_b
0,1444c08e64d55fb3c25f0f09c07ffcf2,2b402d5dc42554061f8ea98d1916f148,BA,lauro de freitas,irece
1,1444c08e64d55fb3c25f0f09c07ffcf2,398cb257329ef7af7f1943a8974a3cbc,BA,lauro de freitas,salvador
2,1444c08e64d55fb3c25f0f09c07ffcf2,4221a7df464f1fe2955934e30ff3a5a1,BA,lauro de freitas,bahia
3,1444c08e64d55fb3c25f0f09c07ffcf2,43753b27d77860f1654aa72e251a7878,BA,lauro de freitas,barro alto
4,1444c08e64d55fb3c25f0f09c07ffcf2,4aba391bc3b88717ce08eb11e44937b2,BA,lauro de freitas,arraial d'ajuda (porto seguro)
5,1444c08e64d55fb3c25f0f09c07ffcf2,4fb41dff7c50136976d1a5cf004a42e2,BA,lauro de freitas,feira de santana
6,1444c08e64d55fb3c25f0f09c07ffcf2,651530bf5c607240ccdd89a30c9c9712,BA,lauro de freitas,ipira
7,1444c08e64d55fb3c25f0f09c07ffcf2,659e8466eb3ff1b0e8740d74fb7bbedd,BA,lauro de freitas,eunapolis
8,1444c08e64d55fb3c25f0f09c07ffcf2,75d34ebb1bd0bd7dde40dd507b8169c3,BA,lauro de freitas,salvador
9,1444c08e64d55fb3c25f0f09c07ffcf2,a3dd39f583bc80bd8c5901c95878921e,BA,lauro de freitas,salvador


**Інтерпретація:** показано **20** пар. Умова `s1.seller_id < s2.seller_id` прибирає self-пари та дзеркальні дублікати.

## JOIN 5 - Anti-join
**Business-question:** які orders не мають жодного review?

In [42]:
q5=run_query("""
SELECT o.order_id,o.customer_id,o.order_status,o.order_purchase_timestamp
FROM olist_orders o LEFT JOIN olist_order_reviews rv ON rv.order_id=o.order_id
WHERE rv.review_row_id IS NULL ORDER BY o.order_purchase_timestamp LIMIT 20
"""); display(q5)
n=scalar("SELECT COUNT(*) FROM olist_orders o WHERE NOT EXISTS(SELECT 1 FROM olist_order_reviews rv WHERE rv.order_id=o.order_id)")
display(Markdown(f"**Інтерпретація:** orders без review: **{n}**. Таблиця показує перші 20."))

,order_id,customer_id,order_status,order_purchase_timestamp
0,cda873529ca7ab71f677d5ec11a40304,76c74aaff2f3f7355f46d9818ad092b8,shipped,2016-10-05 16:57:30
1,5da2004d85f2828349d887c989da2adc,95f592086150f0c95d3082aea047b816,delivered,2016-10-05 20:20:59
2,02190241f7190a1f3c7e0df95a749c6a,55c9044401ffab0c717f24405e2e042f,delivered,2016-10-07 10:52:22
3,0efd0bc268d34da3f01f4ff25a6d4335,95446917717bb58d553d107d0f1668f6,shipped,2016-10-07 15:53:31
4,c776863a93dc0740c6e7d78104b21413,1710b798ebdc5db6665c815b90a64999,delivered,2016-10-10 15:01:27
5,d730d8d0ea3149120703933fdf98b2d4,ce2a66464da8b80dedd7639a6e489a58,shipped,2017-01-13 20:12:41
6,05cd156f0f47579a36d7a52724fc31b8,976224a95faf328b17654b1b1411d5e1,delivered,2017-01-14 18:27:28
7,719b940ab2f1e351ffb14a0a02ffe3c4,ede9a1daf58d256c0d648ef7e1479010,delivered,2017-01-16 13:01:51
8,fbd82b1fe98967de0db29b6cf134d00a,45f5c9d984dbfa2105e3544cc85133bf,delivered,2017-01-18 08:55:36
9,171398a0cd843c65ac78aee9e05a1f49,f0244902c07b7c0799e468218579a925,delivered,2017-01-18 16:07:37


**Інтерпретація:** orders без review: **768**. Таблиця показує перші 20.

# 9. GROUP BY, HAVING та агрегати

## Агрегат 1 - Revenue per category
**Business-question:** які категорії мають gross revenue понад 10 000?

In [43]:
a1=run_query("""
SELECT p.product_category_name,COUNT(DISTINCT oi.order_id) n_orders,SUM(oi.price)::NUMERIC(12,2) gross_revenue,
SUM(oi.freight_value)::NUMERIC(12,2) freight,AVG(oi.price)::NUMERIC(8,2) avg_item_price
FROM olist_order_items oi JOIN olist_products p ON p.product_id=oi.product_id
GROUP BY p.product_category_name HAVING SUM(oi.price)>10000 ORDER BY gross_revenue DESC NULLS LAST
"""); display(a1)
if len(a1): display(Markdown(f"**Інтерпретація:** HAVING пройшли **{len(a1)}** категорій. Лідер - **{a1.iloc[0].product_category_name}**, revenue **{a1.iloc[0].gross_revenue}**."))

,product_category_name,n_orders,gross_revenue,freight,avg_item_price
0,beleza_saude,8836,1258681.34,182566.73,130.16
1,relogios_presentes,5624,1205005.68,100535.93,201.14
2,cama_mesa_banho,9417,1036988.68,204693.04,93.30
3,esporte_lazer,7720,988048.97,168607.51,114.34
4,informatica_acessorios,6689,911954.32,147318.08,116.51
5,moveis_decoracao,6449,729762.49,172749.30,87.56
6,cool_stuff,3632,635290.85,84039.10,167.36
7,utilidades_domesticas,5884,632248.66,146149.11,90.79
8,automotivo,3897,592720.11,92664.21,139.96
9,ferramentas_jardim,3518,485256.46,98962.75,111.63


**Інтерпретація:** HAVING пройшли **53** категорій. Лідер - **beleza_saude**, revenue **1258681.34**.

## Агрегат 2 - AOV per customer state
**Business-question:** де найбільший AOV серед states з щонайменше 100 orders?

In [44]:
a2=run_query("""
WITH ot AS (
 SELECT o.order_id,c.customer_state,SUM(oi.price+oi.freight_value)::NUMERIC(12,2) order_total
 FROM olist_orders o JOIN olist_customers c ON c.customer_id=o.customer_id
 JOIN olist_order_items oi ON oi.order_id=o.order_id GROUP BY o.order_id,c.customer_state
)
SELECT customer_state,COUNT(*) n_orders,SUM(order_total)::NUMERIC(12,2) gmv,
(SUM(order_total)/NULLIF(COUNT(*),0))::NUMERIC(8,2) aov
FROM ot GROUP BY customer_state HAVING COUNT(*)>=100 ORDER BY aov DESC NULLS LAST
"""); display(a2)
if len(a2): display(Markdown(f"**Інтерпретація:** після HAVING залишилося **{len(a2)}** states. Найвищий AOV - **{a2.iloc[0].customer_state}: {a2.iloc[0].aov}**."))

,customer_state,n_orders,gmv,aov
0,PB,532,140987.81,265.01
1,AL,411,96229.40,234.13
2,RO,247,57558.02,233.03
3,PA,970,217647.11,224.38
4,TO,279,61354.42,219.91
5,PI,493,108132.28,219.34
6,SE,345,73032.32,211.69
7,RN,482,101895.08,211.40
8,CE,1327,275606.30,207.69
9,MT,903,186168.96,206.17


**Інтерпретація:** після HAVING залишилося **24** states. Найвищий AOV - **PB: 265.01**.

## Агрегат 3 - STRING_AGG категорій customer
**Business-question:** які категорії купував кожен реальний buyer і скільки він витратив?

In [45]:
a3=run_query("""
SELECT c.customer_unique_id,COUNT(DISTINCT o.order_id) n_orders,SUM(oi.price)::NUMERIC(12,2) total_spend,
STRING_AGG(DISTINCT COALESCE(p.product_category_name,'(unknown)'),', ' ORDER BY COALESCE(p.product_category_name,'(unknown)')) categories
FROM olist_customers c JOIN olist_orders o ON o.customer_id=c.customer_id
JOIN olist_order_items oi ON oi.order_id=o.order_id LEFT JOIN olist_products p ON p.product_id=oi.product_id
GROUP BY c.customer_unique_id HAVING COUNT(DISTINCT o.order_id)>=1
ORDER BY total_spend DESC NULLS LAST LIMIT 20
"""); display(a3)
if len(a3): display(Markdown(f"**Інтерпретація:** для `{a3.iloc[0].customer_unique_id}` STRING_AGG зібрав categories в один список. Total spend **{a3.iloc[0].total_spend}**. WHERE фільтрує rows до агрегації, HAVING - groups після агрегації."))

,customer_unique_id,n_orders,total_spend,categories
0,0a0a92112bd4c708ca5fde585afaa872,1,13440.00,telefonia_fixa
1,da122df9eeddfedc1dc1f5349a1a690c,2,7388.00,eletroportateis
2,763c8b1c9c68a0229c42c9fc6f662b93,1,7160.00,telefonia_fixa
3,dc4802a71eae9be1dd28f5d788ceb526,1,6735.00,utilidades_domesticas
4,459bef486812aa25204be022145caa62,1,6729.00,pcs
5,ff4159b92c40ebe40454e3e6a7c35ed6,1,6499.00,artes
6,4007669dec559734d6f53e029e360987,1,5934.60,agro_industria_e_comercio
7,eebb5dda148d3893cdaf5b5ca3040ccb,1,4690.00,eletroportateis
8,5d0a2980b292d049061542014e8960bf,1,4599.90,cool_stuff
9,48e1ac109decbb87765a3eade6854098,1,4590.00,pcs


**Інтерпретація:** для `0a0a92112bd4c708ca5fde585afaa872` STRING_AGG зібрав categories в один список. Total spend **13440.0**. WHERE фільтрує rows до агрегації, HAVING - groups після агрегації.

# 10. Set operation
**Business-question:** у яких states є і customers, і sellers? Використовується `INTERSECT`, бо потрібен перетин двох наборів.

In [46]:
set_q=run_query("""
SELECT customer_state state FROM olist_customers
INTERSECT
SELECT seller_state state FROM olist_sellers
ORDER BY state
"""); display(set_q)
display(Markdown(f"**Інтерпретація:** INTERSECT повернув **{len(set_q)}** спільних states."))

,state
0,AC
1,AM
2,BA
3,CE
4,DF
5,ES
6,GO
7,MA
8,MG
9,MS


**Інтерпретація:** INTERSECT повернув **23** спільних states.

# 11. Window function
**Business-question:** яке найновіше замовлення має кожен реальний buyer?

In [47]:
wq=run_query("""
WITH ranked_orders AS (
 SELECT c.customer_unique_id,o.customer_id,o.order_id,o.order_purchase_timestamp,
 ROW_NUMBER() OVER(PARTITION BY c.customer_unique_id ORDER BY o.order_purchase_timestamp DESC,o.order_id DESC) rn
 FROM olist_orders o JOIN olist_customers c ON c.customer_id=o.customer_id
)
SELECT customer_unique_id,customer_id,order_id,order_purchase_timestamp
FROM ranked_orders WHERE rn=1 ORDER BY customer_unique_id LIMIT 20
"""); display(wq)
display(Markdown(f"**Інтерпретація:** показано перші **{len(wq)}** latest-order rows. `rn=1` вибирається у зовнішньому запиті, бо WHERE виконується до window function."))

,customer_unique_id,customer_id,order_id,order_purchase_timestamp
0,0000366f3b9a7992bf8c76cfdf3221e2,fadbb3709178fc513abc1b2670aa1ad2,e22acc9c116caa3f2b7121bbb380d08e,2018-05-10 10:56:27
1,0000b849f77a49e4a4ce2b2a4ca5be3f,4cb282e167ae9234755102258dd52ee8,3594e05a005ac4d06a72673270ef9ec9,2018-05-07 11:11:27
2,0000f46a3911fa3c0805444483337064,9b3932a6253894a02c1df9d19004239f,b33ec3b699337181488304f362a6b734,2017-03-10 21:05:03
3,0000f6ccb0745a6a4b88665a16c9f078,914991f0c02ef0843c0e7010c819d642,41272756ecddd9a9ed0180413cc22fb6,2017-10-12 20:29:41
4,0004aac84e0df4da2b147fca70cf8255,47227568b10f5f58a524a75507e6992c,d957021f1127559cd947b62533f484f7,2017-11-14 19:45:42
5,0004bd2a26a76fe21f786e4fbd80607f,4a913a170c26e3c8052ed0202849b5a8,3e470077b690ea3e3d501cffb5e0c499,2018-04-05 19:33:16
6,00050ab1314c0e55a6ca13cf7181fecf,d2509c13692836fc0449e88cf9eb4858,d0028facea13f508e880202d7097a5a1,2018-04-20 12:57:23
7,00053a61a98854899e70ed204dd4bafe,a81ebb9b32f102298c0c89635b4b3154,44e608f2db00c74a1fe329de44416a4e,2018-02-28 11:15:41
8,0005e1862207bf6ccc02e4228effd9a0,3b37fb626fdf46cd99d37ec62afa88ff,ae76bef74b97bcb0b3e355e60d9a6f9c,2017-03-04 23:32:12
9,0005ef4cd20d2893f0d9fbd94d3c0d97,c59e8ff99836e90d8b457d4122dc34e9,01b330808c5819a6a3cb79b72f0b8288,2018-03-12 15:22:12


**Інтерпретація:** показано перші **20** latest-order rows. `rn=1` вибирається у зовнішньому запиті, бо WHERE виконується до window function.

# 12. Reflection

Найскладнішим для мене був `FULL OUTER JOIN`, тому що тут потрібно враховувати не лише збіги, а й рядки, які існують тільки в одному з двох наборів. Найскладніше було сформулювати business-question для `SELF JOIN`: технічно порівнювати продавців між собою просто, але потрібно пояснити бізнес-сенс такого порівняння. Найкориснішою PostgreSQL-фічею для production ETL я вважаю `ON CONFLICT`, оскільки вона дозволяє робити повторювані завантаження та upsert без створення дублікатів. `RETURNING` додатково зручний для audit-output.

У реальному ML-проєкті window functions я б застосовував для вибору останньої події користувача, дедуплікації, побудови sequence features та Top-N ознак. У Olist є типові data-quality особливості: не всі замовлення мають reviews, частина товарів може не мати категорії, а review-тексти часто порожні. Такі пропуски не завжди є помилкою і потребують окремої семантики.

Для customer-level ML-фіч важливо використовувати `customer_unique_id`, а не `customer_id`, бо один реальний покупець може мати різні `customer_id` у різних замовленнях. Інакше повторні покупки розпадуться на окремі рядки, а frequency буде занижена. Перед `seller_score` також потрібно перейти до грануляності `seller_id x order_id`. Якщо напряму з'єднати reviews з `order_items`, один order-level review повториться за кількістю позицій замовлення і спотворить seller rating.